# A. Importando bibliotecas

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
import matplotlib.pyplot as plt

# B. BASES

## Base Desmatamento

In [2]:
df_desmatAm = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/tabela_de_desmatamento_ucs_atualizado_prodes-2023(1).xlsx', skiprows= 1, skipfooter=1)
df_desmatAm.drop(['TOTAL', 'Até 2007'], axis=1, inplace=True)
df_desmatMA = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/tabela_de_desmatamento_ucs_atualizado_prodes-2023(1).xlsx', skiprows= 1, skipfooter=1, sheet_name = 'Mata Atlântica')
df_desmatMA.drop(['TOTAL', 'Até 2000', '2001 a 2004',2006], axis=1, inplace=True)
df_desmatPp = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/tabela_de_desmatamento_ucs_atualizado_prodes-2023(1).xlsx', skiprows= 1, skipfooter=1, sheet_name = 'Pampa')
df_desmatPp.drop(['TOTAL', 'Até 2000', '2001 a 2004',2006], axis=1, inplace=True)
df_desmatCe = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/tabela_de_desmatamento_ucs_atualizado_prodes-2023(1).xlsx', skiprows= 1, skipfooter=1, sheet_name = 'Cerrado')
df_desmatCe.drop(['TOTAL', 'Até 2000', 2002,2004,2006], axis=1, inplace=True)
df_desmatCa = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/tabela_de_desmatamento_ucs_atualizado_prodes-2023(1).xlsx', skiprows= 1, skipfooter=1, sheet_name = 'Caatinga')
df_desmatCa.drop(['TOTAL', 'Até 2000', '2001 a 2004',2006], axis=1, inplace=True)



In [3]:
df_desmat_pre = pd.concat([df_desmatAm, df_desmatMA, df_desmatPp, df_desmatCe, df_desmatCa], ignore_index=True)
df_desmat_pre = df_desmat_pre.fillna(0)
df_desmat_pre.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 321 entries, 0 to 320
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cnuc                    321 non-null    object 
 1   unidade de conservação  321 non-null    object 
 2   área_ha_uc              321 non-null    float64
 3   2008                    321 non-null    float64
 4   2009                    321 non-null    float64
 5   2010                    321 non-null    float64
 6   2011                    321 non-null    float64
 7   2012                    321 non-null    float64
 8   2013                    321 non-null    float64
 9   2014                    321 non-null    float64
 10  2015                    321 non-null    float64
 11  2016                    321 non-null    float64
 12  2017                    321 non-null    float64
 13  2018                    321 non-null    float64
 14  2019                    321 non-null    fl

In [4]:
#Tratando duplicatas
duplicatas = df_desmat_pre[df_desmat_pre.duplicated('cnuc', keep=False)]

# Função para consolidar registros duplicados
def consolidar_duplicatas(grupo):
    consolidado = {
        'cnuc': grupo['cnuc'].iloc[1],  # Mantém o mesmo 'cnuc'
        'unidade de conservação': grupo['unidade de conservação'].iloc[1],  # Mantém o segundo nome
        'área_ha_uc': grupo['área_ha_uc'].iloc[1],  # Mantém a segunda área
    }
    
    # Consolida os valores das colunas de anos (soma)
    for ano in range(2008, 2024):
        coluna = ano  # Mantém o ano como número
        if coluna in grupo.columns:
            consolidado[coluna] = grupo[coluna].sum()
    return pd.Series(consolidado)

# Verificando duplicatas
duplicatas = df_desmat_pre[df_desmat_pre.duplicated('cnuc', keep=False)]#
if not duplicatas.empty:
    df_consolidado = duplicatas.groupby('cnuc').apply(consolidar_duplicatas).reset_index(drop=True)
    
    # Removendo as duplicatas do DataFrame original
    df_sem_duplicatas = df_desmat_pre.drop(duplicatas.index)
    
    # Adicionando os registros consolidados ao DataFrame original
    df_desmat = pd.concat([df_sem_duplicatas, df_consolidado], ignore_index=True)#
# Exibindo informações do DataFrame consolidado
df_desmat.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cnuc                    304 non-null    object 
 1   unidade de conservação  304 non-null    object 
 2   área_ha_uc              304 non-null    float64
 3   2008                    304 non-null    float64
 4   2009                    304 non-null    float64
 5   2010                    304 non-null    float64
 6   2011                    304 non-null    float64
 7   2012                    304 non-null    float64
 8   2013                    304 non-null    float64
 9   2014                    304 non-null    float64
 10  2015                    304 non-null    float64
 11  2016                    304 non-null    float64
 12  2017                    304 non-null    float64
 13  2018                    304 non-null    float64
 14  2019                    304 non-null    fl

## Base SAMGe

In [5]:
excel_file = pd.ExcelFile('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/Dados%20do%20SAMGe_completo.xlsx')
planilhas = excel_file.sheet_names
print("Planilhas encontradas:")
for planilha in planilhas:
    print(f"- {planilha}")

Planilhas encontradas:
- 0 - Atualização
- 1 - Identificação e Resultados
- 2 - Recursos e Valores (RV)
- 3 - Usos
- 4 - Ações
- 5 - Ações x Usos
- 6 - RV x Ações x Usos
- 7 - Processos Prioritários


### SAMGe 1: Identificação e Resultados 

In [6]:
df_samge1 = pd.read_excel('https://github.com/JuPLopez/JL_mba_enap/raw/refs/heads/main/Estatistica_Descritiva/Dados%20do%20SAMGe_completo.xlsx', sheet_name = '1 - Identificação e Resultados')
df_samge1.rename(columns={'CNUC':'cnuc'}, inplace=True)
df_samge1 = df_samge1[df_samge1["Ano"] == 2023]
df_samge1 = df_samge1[df_samge1["Esfera Administrativa"] == 'Federal']

df_samge1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 329 entries, 0 to 328
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID Entrada                      329 non-null    int64  
 1   Ano                             329 non-null    int64  
 2   Esfera Administrativa           329 non-null    object 
 3   cnuc                            329 non-null    object 
 4   Nome da UC                      329 non-null    object 
 5   Gerência Regional/Órgão Gestor  329 non-null    object 
 6   NGI                             219 non-null    object 
 7   UF                              329 non-null    object 
 8   Municípios Abrangidos           329 non-null    object 
 9   Latitude                        329 non-null    float64
 10  Longitude                       329 non-null    float64
 11  Quem Preenche                   329 non-null    object 
 12  Plano de Manejo                 329 non-n

In [7]:
df_samge1["Ano"].unique()

array([2023], dtype=int64)

## MESCLAR BASES

In [8]:
df_identif = df_samge1[['cnuc', 'Nome da UC', 'UF', 'Plano de Manejo', 'Categoria (Sigla)', 'Grupo (Sigla)', 'Bioma', 'Ano de Criação']]

df_baseED = pd.merge(df_desmat, df_identif, on='cnuc', how='inner') 
#df_baseED['Ano de Criação'] = df_baseED['Ano de Criação'].astype(int)
df_baseED.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cnuc                    297 non-null    object 
 1   unidade de conservação  297 non-null    object 
 2   área_ha_uc              297 non-null    float64
 3   2008                    297 non-null    float64
 4   2009                    297 non-null    float64
 5   2010                    297 non-null    float64
 6   2011                    297 non-null    float64
 7   2012                    297 non-null    float64
 8   2013                    297 non-null    float64
 9   2014                    297 non-null    float64
 10  2015                    297 non-null    float64
 11  2016                    297 non-null    float64
 12  2017                    297 non-null    float64
 13  2018                    297 non-null    float64
 14  2019                    297 non-null    fl

In [9]:
df_baseED['TotalDesmat'] = df_baseED[2008] + df_baseED[2009]+ df_baseED[2010]+ df_baseED[2011]+ df_baseED[2012]+ df_baseED[2013]+ df_baseED[2014]+ df_baseED[2015]+ df_baseED[2016]+ df_baseED[2017]+ df_baseED[2018]+ df_baseED[2019]+ df_baseED[2020]+ df_baseED[2021]+ df_baseED[2022]+ df_baseED[2023]
df_baseED['%Desmat'] = df_baseED['TotalDesmat']/df_baseED['área_ha_uc']*100
df_baseED.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297 entries, 0 to 296
Data columns (total 28 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cnuc                    297 non-null    object 
 1   unidade de conservação  297 non-null    object 
 2   área_ha_uc              297 non-null    float64
 3   2008                    297 non-null    float64
 4   2009                    297 non-null    float64
 5   2010                    297 non-null    float64
 6   2011                    297 non-null    float64
 7   2012                    297 non-null    float64
 8   2013                    297 non-null    float64
 9   2014                    297 non-null    float64
 10  2015                    297 non-null    float64
 11  2016                    297 non-null    float64
 12  2017                    297 non-null    float64
 13  2018                    297 non-null    float64
 14  2019                    297 non-null    fl

In [10]:
nome_arquivo = 'df_baseED.xlsx'
df_baseED.to_excel(nome_arquivo)